In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Extraction datasets inspection

Run and inspect every structured-extraction dataset used by the MixedDecoder training
(`s_03_11_train_mixed_decoder.py`), from `cite` through `sql`:

| type | module | task |
|------|--------|------|
| `cite` | `mllm.train.encdec_graph_bert` | masked-citation recall (anchor span) |
| `keyval` | `mllm.train.key_val_recall` | key → value recall |
| `jsonfield` | `mllm.train.json_field_recall` | JSON path → field value |
| `jsonata` | `mllm.train.jsonata_recall` | JSONata/jq selection + transform |
| `xmlxpath` | `mllm.train.xml_xpath_recall` | XML/XPath → node value |
| `sql` | `mllm.train.sql_select_recall` | SQL select/aggregate |

For each we build one batch, **decode the tokens** (encoder record, prompt, target),
and verify the target value is actually present in the encoded context (the
“needle-in-context” forcing function the datasets are designed around).

In [10]:
import sys
from typing import List, Optional
if '..' not in sys.path: sys.path.append('..')

import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer

from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader
from mllm.train.key_val_recall import KeyValRecallCfg, KeyValRecallDataset, create_key_val_recall_dataloader
from mllm.train.json_field_recall import JsonFieldRecallCfg, JsonFieldRecallDataset, create_json_field_recall_dataloader
from mllm.train.jsonata_recall import JsonataRecallCfg, JsonataRecallDataset, create_jsonata_recall_dataloader
from mllm.train.xml_xpath_recall import XmlXpathRecallCfg, XmlXpathRecallDataset, create_xml_xpath_recall_dataloader
from mllm.train.sql_select_recall import SqlSelectRecallCfg, SqlSelectRecallDataset, create_sql_select_recall_dataloader

## Config

We use a small in-memory Wikipedia-like corpus so the notebook is fully self-contained
(no multi-GB download). The recall builders only need *real words / spans* drawn from
`text`, so a handful of rich paragraphs is enough to exercise every code path.

Encoder and decoder tokenizer are both `bert-base-uncased` (shared vocab) so decoded
tokens line up exactly between the encoder record and the decoder prompt/target.

In [11]:
device = torch.device('cpu')
max_seq_len = 192
batch_size = 3
n_special_toks = 1000
seed = 42

tkz_enc = AutoTokenizer.from_pretrained('bert-base-uncased')
tkz_dec = AutoTokenizer.from_pretrained('bert-base-uncased')
if tkz_dec.pad_token is None:
    tkz_dec.pad_token = tkz_dec.eos_token
print('enc vocab:', len(tkz_enc), '| dec vocab:', len(tkz_dec))

enc vocab: 30522 | dec vocab: 30522


In [12]:
SAMPLE_TEXTS = [
    (
        'The Amazon River in South America is the largest river by discharge volume of water '
        'in the world. It has a length of about 6400 kilometres and drains an area of roughly '
        'seven million square kilometres. The river begins in the Andes mountains of Peru and '
        'flows eastward across Brazil before emptying into the Atlantic Ocean. Hundreds of '
        'tributaries feed the main channel, including the Negro, the Madeira and the Tapajos.'
    ),
    (
        'Photosynthesis is the process used by plants, algae and some bacteria to convert light '
        'energy into chemical energy stored in glucose. Chlorophyll inside the chloroplasts '
        'absorbs sunlight, water is taken up by the roots and carbon dioxide enters through tiny '
        'pores called stomata. The overall reaction releases oxygen as a byproduct, supporting '
        'most aerobic life on Earth and forming the base of nearly every food chain.'
    ),
    (
        'The Eiffel Tower is a wrought iron lattice tower located on the Champ de Mars in Paris, '
        'France. It was designed by the engineer Gustave Eiffel and completed in 1889 as the '
        'entrance arch to the Worlds Fair. Standing about 330 metres tall, it was the tallest '
        'man made structure in the world until the Chrysler Building was built in New York in 1930. '
        'Millions of visitors climb its three levels every year.'
    ),
    (
        'A black hole is a region of spacetime where gravity is so strong that nothing, including '
        'light, can escape from it. The boundary of no return is called the event horizon. Black '
        'holes form when massive stars collapse at the end of their life cycles. The supermassive '
        'black hole at the centre of our galaxy is known as Sagittarius A star and has a mass of '
        'about four million times that of the Sun.'
    ),
    (
        'The printing press was invented by Johannes Gutenberg in the German city of Mainz around '
        '1440. By using movable metal type, it allowed books to be produced quickly and cheaply '
        'compared with manuscripts copied by hand. The technology spread rapidly across Europe and '
        'is widely credited with accelerating the Renaissance, the Reformation and the scientific '
        'revolution that followed in later centuries.'
    ),
    (
        'Mount Everest, located in the Himalayas on the border between Nepal and Tibet, is the '
        'highest mountain above sea level on Earth. Its summit reaches an elevation of 8849 metres. '
        'The mountain was first climbed in 1953 by Edmund Hillary of New Zealand and Tenzing Norgay, '
        'a Sherpa of Nepal. Each climbing season many expeditions attempt the dangerous ascent.'
    ),
]

ds = Dataset.from_dict({'text': SAMPLE_TEXTS})
print('corpus docs:', len(ds))

corpus docs: 6


## Decoding helpers

Each dataset yields a `MaskedCiteBatch`. The tensors we care about:

* `inp_toks` / `inp_masked_toks` — the encoder record (**encoder** vocab).
* `prompts_toks` — the query the decoder is conditioned on (**decoder** vocab).
* `cites_toks` — the target the decoder must produce (**decoder** vocab).

Per-sample `tokens_subsets` also carry the raw `prompt` string plus the encoder-vocab
value (`toks_cite`) which we use to confirm the needle is inside the record.

In [13]:
def _decode_row(toks_row: torch.Tensor, att_row: Optional[torch.Tensor], tkz, skip_special=False) -> str:
    if att_row is not None:
        n = int(att_row.sum().item())
        ids = toks_row[:n].tolist()
    else:
        ids = toks_row.tolist()
    return tkz.decode(ids, skip_special_tokens=skip_special)


def _contains_subseq(haystack: List[int], needle: List[int]) -> bool:
    if not needle:
        return True
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i:i + m] == needle:
            return True
    return False


def show_batch(name: str, batch, n: int = 3):
    print('=' * 100)
    print(f'### {name}  |  batch_size={batch.inp_toks.shape[0]}  enc_len={batch.inp_toks.shape[1]}  '
          f'dec_prompt_len={batch.prompts_toks.shape[1]}  dec_cite_len={batch.cites_toks.shape[1]}')
    print('=' * 100)
    n = min(n, batch.inp_toks.shape[0])
    for i in range(n):
        sub = batch.tokens_subsets[i]
        record = _decode_row(batch.inp_toks[i], batch.inp_att_mask[i], tkz_enc)
        masked = _decode_row(batch.inp_masked_toks[i], batch.inp_att_mask[i], tkz_enc)
        prompt = _decode_row(batch.prompts_toks[i], batch.prompts_att_mask[i], tkz_dec)
        target = _decode_row(batch.cites_toks[i], batch.cites_att_mask[i], tkz_dec)
        # Needle-in-context check: encoder-vocab target value must live inside the record.
        needle_in_ctx = _contains_subseq(list(sub.toks_inp), list(sub.toks_cite))
        print(f'\n--- sample {i} ---')
        print(f'  ENC RECORD : {record}')
        if masked != record:
            print(f'  ENC MASKED : {masked}')
        print(f'  PROMPT     : {prompt}')
        print(f'  TARGET     : {target}')
        print(f'  needle in context: {needle_in_ctx}')

## 1. `cite` — masked citation recall

In [21]:
cite_ds = MaskedCiteDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    mask_cfg=None, prompt_all=False, device=device, tkz_dec=tkz_dec,
)
cite_ds_loader = create_masked_cite_dataloader(cite_ds, batch_size=batch_size)
cite_ds.shuffle(seed=seed)
cite_batch = next(cite_ds_loader)
cite_batch = next(cite_ds_loader)
show_batch('cite', cite_batch)

R0. Create MaskedCiteDataset dataloader. batch_size=3. shuffle=True.
R0. Shuffle dataset
### cite  |  batch_size=3  enc_len=98  dec_prompt_len=27  dec_cite_len=49

--- sample 0 ---
  ENC RECORD : [CLS] the printing press was invented by johannes gutenberg in the german city of mainz around 1440. by using 陽 shadow kern movable metal type, it allowed books to be produced quickly and cheaply compared with manuscripts copied by hand. the technology spread rapidly across europe and is widely credited with accelerating the renaissance emily eddiecchi, the reformation and the scientific revolution that followed in later centuries. [SEP]
  PROMPT     : cite tag begin : " 陽 shadow kern ". cite tag end : " emily eddiecchi ". produce output text between these tags.
  TARGET     : movable metal type, it allowed books to be produced quickly and cheaply compared with manuscripts copied by hand. the technology spread rapidly across europe and is widely credited with accelerating the renaissance
  nee

## 2. `keyval` — key → value recall

In [22]:
keyval_cfg = KeyValRecallCfg(min_pairs=4, max_pairs=12, value_max_words=3)
keyval_ds = KeyValRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=keyval_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
keyval_ds_loader = create_key_val_recall_dataloader(keyval_ds, batch_size=batch_size)
keyval_ds.shuffle(seed=seed)
keyval_batch = next(keyval_ds_loader)
show_batch('keyval', keyval_batch)

R0. Create KeyValRecallDataset dataloader. batch_size=3.
p: 0 key_cands: 52 words: 80 budget: 190 used_len: 0
key_ids: [2003]
value_ids: [1010, 2064, 4019]
p: 1 key_cands: 52 words: 80 budget: 190 used_len: 5
key_ids: [2038]
value_ids: [3340]
p: 2 key_cands: 52 words: 80 budget: 190 used_len: 9
key_ids: [2004]
value_ids: [8992, 2003, 2061]
p: 3 key_cands: 52 words: 80 budget: 190 used_len: 15
key_ids: [1997]
value_ids: [2422, 1010]
p: 4 key_cands: 52 words: 80 budget: 190 used_len: 20
key_ids: [3565, 9335, 12742]
value_ids: [1997, 2055, 2176]
p: 5 key_cands: 52 words: 80 budget: 190 used_len: 28
key_ids: [2176]
value_ids: [2061]
p: 6 key_cands: 52 words: 80 budget: 190 used_len: 32
key_ids: [2164]
value_ids: [1037]
p: 7 key_cands: 52 words: 80 budget: 190 used_len: 36
key_ids: [3742]
value_ids: [2003, 2170, 1996]
p: 8 key_cands: 52 words: 80 budget: 190 used_len: 42
key_ids: [2170]
value_ids: [4920]
p: 0 key_cands: 56 words: 79 budget: 190 used_len: 0
key_ids: [2081]
value_ids: [6478, 

## 3. `jsonfield` — JSON path → field value

In [16]:
jsonfield_cfg = JsonFieldRecallCfg(
    min_fields=4, max_fields=10, max_depth=3, max_array_len=4, value_max_words=3,
)
jsonfield_ds = JsonFieldRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=jsonfield_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
jsonfield_ds.shuffle(seed=seed)
jsonfield_batch = next(create_json_field_recall_dataloader(jsonfield_ds, batch_size=batch_size))
show_batch('jsonfield', jsonfield_batch)

R0. Create JsonFieldRecallDataset dataloader. batch_size=3.
### jsonfield  |  batch_size=3  enc_len=118  dec_prompt_len=17  dec_cite_len=4

--- sample 0 ---
  ENC RECORD : [CLS] { " return " : " return is called " } [SEP]
  PROMPT     : what is the value at path return? return value only.
  TARGET     : return is called [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] { " building " : { " about " : " levels every ", " tallest " : [ " chrysler ", 372733, " was ", " champ de mars " ] }, " by " : [ " millions of " ], " champ " : [ 75848, " iron lattice tower " ], " about " : { " it " : " fair. standing ", " metres " : null, " year " : [ " iron lattice tower ", " the entrance arch " ], " paris " : 643279 } } [SEP]
  PROMPT     : what is the value at path building. tallest [ 0 ]? return value only.
  TARGET     : chrysler [SEP]
  needle in context: True

--- sample 2 ---
  ENC RECORD : [CLS] { " in " : " between ", " by " : " highest mountain ", " ascent " : " edmund h

## 4. `jsonata` — JSONata/jq selection + transform

In [17]:
jsonata_cfg = JsonataRecallCfg(
    min_fields=4, max_fields=10, max_depth=3, max_array_len=5,
    value_max_words=3, transform_prob=0.35,
)
jsonata_ds = JsonataRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=jsonata_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
jsonata_ds.shuffle(seed=seed)
jsonata_batch = next(create_jsonata_recall_dataloader(jsonata_ds, batch_size=batch_size))
show_batch('jsonata', jsonata_batch)

R0. Create JsonataRecallDataset dataloader. batch_size=3.
### jsonata  |  batch_size=3  enc_len=98  dec_prompt_len=20  dec_cite_len=4

--- sample 0 ---
  ENC RECORD : [CLS] { " return " : " return is called " } [SEP]
  PROMPT     : [ tier - e ] jq :. return. return value only.
  TARGET     : return is called [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] { " of " : " in ", " levels " : ". standing about ", " engineer " : { " iron " : " levels every year " } } [SEP]
  PROMPT     : [ tier - e ] jq :. engineer. iron. return value only.
  TARGET     : levels every year [SEP]
  needle in context: True

--- sample 2 ---
  ENC RECORD : [CLS] { " everest " : " nepal and ", " highest " : { " in " : [ " an elevation of ", 285902, " on ", ", a " ], " was " : { " norgay " : [ ". ", " and tenzing norgay " ], " summit " : ". each ", " and " : [ " the ", " many ", " season many " ] } } } [SEP]
  PROMPT     : [ tier - e ] jq :. highest. in [ 1 ]. return value only.
  TARGET    

## 5. `xmlxpath` — XML/XPath → node value

In [18]:
xmlxpath_cfg = XmlXpathRecallCfg(
    min_nodes=4, max_nodes=12, max_depth=4, max_children=4, value_max_words=3,
)
xmlxpath_ds = XmlXpathRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=xmlxpath_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
xmlxpath_ds.shuffle(seed=seed)
xmlxpath_batch = next(create_xml_xpath_recall_dataloader(xmlxpath_ds, batch_size=batch_size))
show_batch('xmlxpath', xmlxpath_batch)

R0. Create XmlXpathRecallDataset dataloader. batch_size=3.
### xmlxpath  |  batch_size=3  enc_len=15  dec_prompt_len=21  dec_cite_len=4

--- sample 0 ---
  ENC RECORD : [CLS] < return > return is called < / return > [SEP]
  PROMPT     : xml query : extract / return / text ( ). return value only.
  TARGET     : return is called [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] < champ > it was designed < / champ > [SEP]
  PROMPT     : xml query : extract / champ / text ( ). return value only.
  TARGET     : it was designed [SEP]
  needle in context: True

--- sample 2 ---
  ENC RECORD : [CLS] < norgay > by edmund < / norgay > [SEP]
  PROMPT     : what is the value selected by xpath / norgay / text ( )? return value only.
  TARGET     : by edmund [SEP]
  needle in context: True


## 6. `sql` — SQL select / aggregate

In [19]:
sql_cfg = SqlSelectRecallCfg(
    min_rows=4, max_rows=8, min_cols=3, max_cols=5,
    value_max_words=3, transform_prob=0.30,
)
sql_ds = SqlSelectRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=sql_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
sql_ds.shuffle(seed=seed)
sql_batch = next(create_sql_select_recall_dataloader(sql_ds, batch_size=batch_size))
show_batch('sql', sql_batch)

R0. Create SqlSelectRecallDataset dataloader. batch_size=3.
### sql  |  batch_size=3  enc_len=163  dec_prompt_len=20  dec_cite_len=5

--- sample 0 ---
  ENC RECORD : [CLS] | id | sagittarius | when | horizon | | - - - | - - - | - - - | - - - | | 1 | sun. | their life cycles | 862990 | | 2 | the | a star and | 586491 | | 3 | that of | known as sagittarius | 623312 | | 4 | when massive stars | a star | 517308 | | 5 | is | where gravity | 372733 | | 6 | of | from it. | 916295 | | 7 | of the sun | a | 920285 | | 8 | life | no return is | 89479 | [SEP]
  PROMPT     : [ tier - e ] which when has id 1? return value only.
  TARGET     : their life cycles [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] | id | fair | france | | - - - | - - - | - - - | | 1 | 524301 | 75848 | | 2 | 731202 | 293933 | | 3 | 618337 | 76740 | | 4 | 600708 | 909397 | | 5 | 437844 | 577907 | | 6 | 294244 | 285902 | | 7 | 65380 | 478975 | | 8 | 947032 | 116031 | [SEP]
  PROMPT     : [ tier - e ] sq

## Summary check

Confirm across every dataset that the decoder target value is recoverable from the
encoded context for all samples in the batch (the property that forces the model to
read the context embeddings rather than rely on priors).

In [20]:
all_batches = {
    'cite': cite_batch,
    'keyval': keyval_batch,
    'jsonfield': jsonfield_batch,
    'jsonata': jsonata_batch,
    'xmlxpath': xmlxpath_batch,
    'sql': sql_batch,
}
print(f'{"dataset":<12} {"samples":>8} {"needle_in_ctx":>14}')
print('-' * 36)
for name, b in all_batches.items():
    bs = b.inp_toks.shape[0]
    ok = sum(
        _contains_subseq(list(s.toks_inp), list(s.toks_cite))
        for s in b.tokens_subsets
    )
    print(f'{name:<12} {bs:>8} {f"{ok}/{bs}":>14}')

dataset       samples  needle_in_ctx
------------------------------------
cite                3            3/3
keyval              3            3/3
jsonfield           3            3/3
jsonata             3            3/3
xmlxpath            3            3/3
sql                 3            3/3
